## Imports And Downloads

In [ ]:
!pip install transformers torch langchain pydantic accelerate 

In [2]:
!pip install langchain-core

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch
import json
from pydantic import BaseModel, Field, ValidationError
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

In [4]:
!pip install langchain-community

In [6]:
from langchain_community.llms import HuggingFacePipeline

## Model Loading

In [ ]:
model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    temperature=0.3
)

llm = HuggingFacePipeline(pipeline=pipe)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipython-input-1279001060.py:18: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


## Analysis Class

In [43]:
class ReviewAnalysis(BaseModel):
    sentiment: str = Field(description="positive, negative, or neutral")
    main_issue: str = Field(description="Main problem mentioned in the review")
    rating: int = Field(description="Rating from 1 to 5")
    recommendation: str = Field(description="yes or no")

parser = PydanticOutputParser(pydantic_object=ReviewAnalysis)

## Prompt

In [ ]:
prompt_text = f"""
You are an AI assistant.

Extract structured data from the review using **these exact keys**:

- sentiment: positive, negative, or neutral
- main_issue: main problem mentioned
- rating: integer from 1 to 5
- recommendation: yes or no

Return ONLY a JSON object **with these exact keys**, no extra text, no schema.

Review:
{review_text}
"""

## User Query

In [45]:
review_text = """
I bought this laptop two weeks ago. The performance is good,
but the battery drains very fast and it overheats sometimes.
Not sure if I would buy it again.
"""

In [46]:
formatted_prompt = prompt.format(review=review_text)

raw_output = llm.invoke(prompt_text)

Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Response

In [ ]:
import re

json_match = re.search(r"\{.*\}", raw_output, re.DOTALL)

if json_match:
    raw_json = json_match.group()
    parsed_output = parser.parse(raw_json)
    print(parsed_output.dict())
    
else:
    print("No valid JSON found!")

{'sentiment': 'negative', 'main_issue': 'battery drains very fast and overheats', 'rating': 2, 'recommendation': 'no'}


/tmp/ipython-input-2868004814.py:7: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  print(parsed_output.dict())
